# Homework 8 — Neural networks and deep learning — ML Zoomcamp 2026 cohort

A small CNN in **PyTorch** that classifies curly vs straight hair. The split, seed, transforms, model, loss settings and evaluation procedure are all pinned so the numbers are reproducible. Questions: `homework8_2026.md`.

**Submit at:** https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw08 · platform deadline **Mon 30 Nov**. Target: submit on the Thu 26 Nov 08:00 session.

Kernel: the PyTorch environment from START_HERE (Python 3.11, `torch 2.9.0`, `torchvision 0.24.0`, `numpy 2.3.3`, `Pillow 11.3.0`). Data: `data/train/` and `data/test/` next to this notebook (1,001 images; `test/` is the "held-out evaluation" split).

Scaffold only: question cells are empty.

## Setup — reproducibility (pinned)

Run this before building datasets, loaders, the model or the optimizer. Restart the kernel and run from the top before you read off the final numbers: the seed only reproduces the run if nothing consumed random numbers in between.

In [ ]:
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

# reference_train.py also fixes the CPU thread count (not in homework.md; keeps the float sums in the same order).
torch.set_num_threads(2)

# Must print 2.9.0 / 0.24.0 / 2.3.3 (a '+cpu' suffix or not doesn't matter).
torch.__version__, torchvision.__version__, np.__version__

## Dataset and loaders (pinned)

`ImageFolder` sorts the class folders alphabetically: `curly=0`, `straight=1`. No new random split, and no augmentation on the evaluation images.

In [ ]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Baseline phase: the training images get the same (non-random) transform as the evaluation images.
train_dataset = datasets.ImageFolder('data/train', transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder('data/test', transform=evaluation_transform)

# 800 / 201: the one GIF in data/train/curly is skipped by ImageFolder. Anything else = wrong archive or folder.
assert len(train_dataset) == 800 and len(evaluation_dataset) == 201, (len(train_dataset), len(evaluation_dataset))
assert train_dataset.class_to_idx == {'curly': 0, 'straight': 1}

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

## Model (pinned)

One conv layer (32 filters, 3×3) → max-pool 2×2 → dense 64 → **one logit**. No sigmoid inside the model.

In [ ]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)


model = HairModel().to(device)

## Q1. Loss and output contract

The model emits one raw logit per image. Which loss matches? *(`nn.MSELoss()` / `nn.BCEWithLogitsLoss()` / `nn.CrossEntropyLoss()` / `nn.CosineEmbeddingLoss()`)*

## Loss and optimizer

Put your Q1 loss in `criterion`. The optimizer is pinned. Create both **once**: the augmented phase keeps training this same model with this same optimizer.

In [ ]:
criterion = None  # your Q1 answer

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

## Q2. Parameter count

All trainable parameters, biases included: `sum(p.numel() for p in model.parameters())`. *(896 / 11214912 / 15896912 / 20073473)*

## Baseline training — 10 epochs → `history_baseline.json`

Write a train-one-epoch loop and an evaluate function (you'll reuse both for the augmented phase). Per epoch: `model.train()` → for each batch `zero_grad` → forward → loss → `backward` → `step`; then evaluate the held-out split with `model.eval()` and `torch.no_grad()`. Store four lists: `train_loss`, `train_accuracy`, `evaluation_loss`, `evaluation_accuracy`, and dump them with `json.dump` to `history_baseline.json`.

> Labels arrive as a `long` tensor of shape `(20,)`; the model returns `(20, 1)` floats. `BCEWithLogitsLoss` stops with *"Target size ... must be the same as input size"* (or a dtype error): use `labels.float().unsqueeze(1)`.
> Average per **example**, not per batch: `loss.item() * images.size(0)` summed, divided by the number of images. The 201 evaluation images make 10 batches of 20 plus one batch of **1**, so a mean of batch means is wrong.
> Accuracy: `(torch.sigmoid(logits) >= 0.5)` compared with the float labels.

## Q3. Baseline accuracy

Median of `train_accuracy` in `history_baseline.json`, rounded to 2 decimals (tolerance ±0.05, free number).

## Q4. Baseline loss

**Population** standard deviation of `train_loss`: `np.std(values, ddof=0)`, rounded to 3 decimals (tolerance ±0.02).

> `pd.Series(values).std()` and `statistics.stdev` use `ddof=1` (sample std): not the requested number.

## Data augmentation (pinned transform)

Only the **training** transform changes. Same `model`, same `optimizer`: don't re-create either, and don't re-run the model cell.
The evaluation dataset and loader stay exactly as they are.

> homework.md says nothing about the augmented loader's generator. `reference_train.py` seeds it with `SEED + 1`, pre-filled below so your run follows the reference.

In [ ]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset_augmented = datasets.ImageFolder('data/train', transform=train_transform_augmented)

train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED + 1),
)

## Augmented training — 10 more epochs → `history_augmented.json`

Same loop, `train_loader_augmented` for training, `evaluation_loader` for evaluation. Only these 10 epochs go into `history_augmented.json`.

## Q5. Augmented evaluation loss

Mean of `evaluation_loss` in `history_augmented.json`, rounded to 3 decimals (tolerance ±0.02).

## Q6. Augmented evaluation accuracy

Mean of the **last five** values of `evaluation_accuracy` in `history_augmented.json` (augmented epochs 6–10), rounded to 2 decimals (tolerance ±0.05).

## Cross-check (after you have your own numbers)

`reference_train.py` is the official procedure end to end. From this folder, in the PyTorch environment:

```bash
python reference_train.py --data-dir data
```

It writes `runs/reference/history_baseline.json` and `history_augmented.json` and prints the four Q3–Q6 statistics, so run it only once your own loop is done. Submission: commit this notebook and your two JSON files, and paste the GitHub link as the homework URL.